# D599 Task 3 Market Basket Analysis 

In [5]:
import pandas as pd

# Load the dataset
df = pd.read_excel("Megastore_Dataset_Task_3 3.xlsx", engine='openpyxl')

# Display the first few rows of the dataset
print(df.head())

   OrderID                         ProductName  Quantity         InvoiceDate  \
0   536370         INFLATABLE POLITICAL GLOBE         48 2010-12-01 08:45:00   
1   536370      SET2 RED RETROSPOT TEA TOWELS         18 2010-12-01 08:45:00   
2   536370     PANDA AND BUNNIES STICKER SHEET        12 2010-12-01 08:45:00   
3   536370       RED TOADSTOOL LED NIGHT LIGHT        24 2010-12-01 08:45:00   
4   536370  VINTAGE HEADS AND TAILS CARD GAME         24 2010-12-01 08:45:00   

   UnitPrice   TotalCost         Country DiscountApplied OrderPriority  \
0        0.85        40.8  United States             Yes          High   
1        2.95        53.1  United States             Yes          High   
2        0.85        10.2  United States             Yes          High   
3        1.65        39.6  United States             Yes          High   
4        1.25        30.0  United States             Yes          High   

      Region    Segment ExpeditedShipping PaymentMethod  \
0  Northeast  C

In [42]:
import pandas as pd

# Load the dataset
df = pd.read_excel("Megastore_Dataset_Task_3 3.xlsx", engine='openpyxl')

# Normalize column names to lowercase and strip any leading/trailing spaces
df.columns = df.columns.str.lower().str.strip()

# Verify column names
print("Column names after normalization:")
print(df.columns)

# Step 1: Ordinal Encoding
# Define the order for ordinal variables
order_priority_order = {"Low": 1, "Medium": 2, "High": 3}
customer_satisfaction_order = {
    "Prefer to not respond": 0,
    "Dissatisfied": 1,
    "Very dissatisfied": 2,
    "Satisfied": 3,
    "Very Satisfied": 4,
}

# Apply ordinal encoding
if 'orderpriority' in df.columns:
    df['orderpriority'] = df['orderpriority'].map(order_priority_order)
else:
    print("Column 'orderpriority' not found in the dataset.")

if 'customerordersatisfaction' in df.columns:
    df['customerordersatisfaction'] = df['customerordersatisfaction'].map(customer_satisfaction_order)
else:
    print("Column 'customerordersatisfaction' not found in the dataset.")

# Step 2: One-Hot Encoding
# Apply one-hot encoding for nominal variables
if 'paymentmethod' in df.columns:
    df = pd.get_dummies(df, columns=['paymentmethod'])
else:
    print("Column 'paymentmethod' not found in the dataset.")

# Save the cleaned dataset
cleaned_dataset_path = "cleaned_market_dataset.csv"
df.to_csv(cleaned_dataset_path, index=False)
print(f"Cleaned dataset saved as '{cleaned_dataset_path}'")

# Step 3: Transactionalize the data for market basket analysis
# Ensure the original 'productname' column exists
if 'orderid' in df.columns and 'productname' in df.columns:
    # Group by 'orderid' and aggregate 'productname' into a list
    transactional_data = df.groupby('orderid')['productname'].apply(list)

    # Convert the transactional data into a DataFrame
    transactional_df = transactional_data.reset_index()
    transactional_df.columns = ['orderid', 'items']

    # Display the transactional dataset
    print("Transactional dataset:")
    print(transactional_df.head())

    # Save the transactional dataset for market basket analysis
    transactional_dataset_path = "transactional_dataset.csv"
    transactional_df.to_csv(transactional_dataset_path, index=False)
    print(f"Transactional dataset saved as '{transactional_dataset_path}'")
else:
    print("Columns 'orderid' or 'productname' not found in the dataset.")

Column names after normalization:
Index(['orderid', 'productname', 'quantity', 'invoicedate', 'unitprice',
       'totalcost', 'country', 'discountapplied', 'orderpriority', 'region',
       'segment', 'expeditedshipping', 'paymentmethod',
       'customerordersatisfaction'],
      dtype='object')
Cleaned dataset saved as 'cleaned_market_dataset.csv'
Transactional dataset:
   orderid                                              items
0   536370  [INFLATABLE POLITICAL GLOBE , SET2 RED RETROSP...
1   536852  [POLKADOT RAIN HAT , VINTAGE HEADS AND TAILS C...
2   536974  [EDWARDIAN PARASOL RED, LUNCH BAG RED RETROSPO...
3   537065  [PARTY TIME PENCIL ERASERS, RED RETROSPOT PURS...
4   537463  [PINK POLKADOT CHILDRENS UMBRELLA, RED RETROSP...
Transactional dataset saved as 'transactional_dataset.csv'


In [43]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Load the transactional dataset
transactional_df = pd.read_csv("transactional_dataset.csv")

# Convert the 'items' column into a list of transactions
transactions = transactional_df['items'].apply(eval).tolist()

# Use TransactionEncoder to encode the transactions into a one-hot encoded DataFrame
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
transaction_encoded_df = pd.DataFrame(te_ary, columns=te.columns_)

# Generate frequent itemsets using the Apriori algorithm
frequent_itemsets = apriori(transaction_encoded_df, min_support=0.01, use_colnames=True)

# Generate association rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# Display the association rules
print("Association Rules:")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

# Save the association rules to a CSV file
rules.to_csv("association_rules.csv", index=False)
print("Association rules saved as 'association_rules.csv'")

Association Rules:
                                antecedents  \
0         (CHARLOTTE BAG DOLLY GIRL DESIGN)   
1                      ( DOLLY GIRL BEAKER)   
2               (DOLLY GIRL CHILDRENS BOWL)   
3                      ( DOLLY GIRL BEAKER)   
4                (DOLLY GIRL CHILDRENS CUP)   
...                                     ...   
85169        (PACK OF 6 SKULL PAPER PLATES)   
85170       (SET OF 9 BLACK SKULL BALLOONS)   
85171      (SET OF 9 HEART SHAPED BALLOONS)   
85172  (SET20 RED RETROSPOT PAPER NAPKINS )   
85173        (SET6 RED SPOTTY PAPER PLATES)   

                                             consequents   support  \
0                                   ( DOLLY GIRL BEAKER)  0.011338   
1                      (CHARLOTTE BAG DOLLY GIRL DESIGN)  0.011338   
2                                   ( DOLLY GIRL BEAKER)  0.015873   
3                            (DOLLY GIRL CHILDRENS BOWL)  0.015873   
4                                   ( DOLLY GIRL BEAKER)  0.013605

In [44]:
top_rules = rules.sort_values(by='lift', ascending=False).head(3)
print("Top 3 Relevant Rules:")
print(top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

Top 3 Relevant Rules:
                                             antecedents  \
67670  (SET6 RED SPOTTY PAPER PLATES, SET10 BLUE POLK...   
67663  (SET6 RED SPOTTY PAPER CUPS, SET10 BLUE POLKAD...   
80721  (ALARM CLOCK BAKELIKE IVORY, ALARM CLOCK BAKEL...   

                                             consequents   support  \
67670  (SET10 RED POLKADOT PARTY CANDLES, SET6 RED SP...  0.011338   
67663  (SET10 RED POLKADOT PARTY CANDLES, SET6 RED SP...  0.011338   
80721  (ALARM CLOCK BAKELIKE PINK, CHARLOTTE BAG DOLL...  0.011338   

       confidence  lift  
67670         1.0  88.2  
67663         1.0  88.2  
80721         1.0  88.2  
